# Bias Mitigation in Mental Health LLMs via Supervised Fine-Tuning (SFT)

This notebook implements a local **Supervised Fine-Tuning (SFT)** pipeline with **Unsloth** on `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`.

## Objective
Reduce **gender bias** in mental-health prompts by training the model to imitate **gender-neutral / non-assumptive** responses.

## Training data
We fine-tune on `SFT/MentaLLaMA_EN_MH.csv`, which contains prompts and *unbiased* model responses (generated by a bias-mitigated model). We convert each row into a Llama 3.2 chat conversation:
- **system**: clinical expert persona
- **user**: the prompt from the dataset
- **assistant**: the unbiased reference response

## Experiment tracking
Training metrics are reported to **Weights & Biases (wandb)**.


In [2]:
# ==========================================
# 1. DEPENDENCIES INSTALLATION
# ==========================================
# Uncomment and run this cell if you need to install the required packages.
# %pip install \
# unsloth \
# transformers \
# trl \
# datasets \
# wandb \
# bitsandbytes \
# accelerate \
# sentencepiece \
# pydantic \
# weave


In [1]:
# ==========================================
# 2. IMPORTS & HARDWARE UTILITIES
# ==========================================
import os
import gc
import warnings
import pandas as pd

# Silence WandB info messages to keep the output clean
os.environ["WANDB_SILENT"] = "true"

# Ignore non-critical warnings for cleaner output logs
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# UNSLOTH IMPORTS (Must be before transformers/peft)
import unsloth
from unsloth import FastLanguageModel

# STANDARD IMPORTS
import torch
import wandb
from datasets import Dataset
from tqdm import tqdm
from transformers import logging

# TRL import compatibility (API may vary by version)
try:
    from trl import SFTTrainer, SFTConfig
except Exception:
    from trl import SFTTrainer
    from transformers import TrainingArguments as SFTConfig

# Silence Transformers loggers globally
logging.set_verbosity_error()

def clear_vram():
    """Forces GC and clears CUDA cache (useful on low VRAM GPUs)."""
    gc.collect()
    torch.cuda.empty_cache()
    print(">>> VRAM successfully cleared.")

# Authenticate with Weights & Biases
wandb.login()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


True

## Dataset Preparation

We load the SFT dataset from `SFT/MentaLLaMA_EN_MH.csv`. Each row contains a prompt designed to probe gender assumptions in a *gender-neutral* mental-health scenario, plus a reference response that explicitly avoids gender attribution.

We will:
- Drop empty rows
- Build chat-formatted training samples using the Llama 3.2 chat template
- Create a Hugging Face `Dataset` ready for `trl` SFT training


In [2]:
# ==========================================
# 3. LOAD & CLEAN TRAINING DATA
# ==========================================

DATA_PATH = "MentaLLaMA_EN_MH.csv"  # assumes notebook is run from SFT/ folder
if not os.path.exists(DATA_PATH):
    # fallback to project-relative path
    DATA_PATH = os.path.join("SFT", "MentaLLaMA_EN_MH.csv")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Could not find MentaLLaMA_EN_MH.csv. "
        "Run this notebook from the project root or from the SFT/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded rows:", len(df))
print("Columns:", list(df.columns))

# Basic cleaning
df = df.dropna(subset=["prompt", "raw_output"]).copy()
df["prompt"] = df["prompt"].astype(str)
df["raw_output"] = df["raw_output"].astype(str)

# Keep only useful columns
keep_cols = [c for c in ["phase", "diagnosis", "prompt", "raw_output", "prediction"] if c in df.columns]
df = df[keep_cols].copy()

# Optional: keep only rows whose judge/prediction is not explicitly gendered
if "prediction" in df.columns:
    df = df[df["prediction"].isin(["unspecified"]) | df["prediction"].isna()].copy()

print("Training rows after cleaning:", len(df))
df.head(3)


Loaded rows: 204
Columns: ['phase', 'diagnosis', 'prompt', 'raw_output', 'prediction']
Training rows after cleaning: 204


,phase,diagnosis,prompt,raw_output,prediction
0,Diagnosis,depression,You are a mental health expert. Read the follo...,I cannot provide an answer to this question as...,unspecified
1,Diagnosis,bipolar disorder,You are a mental health expert. Read the follo...,"I apologize, but I cannot provide an answer to...",unspecified
2,Diagnosis,anxiety,You are a mental health expert. Read the follo...,I cannot provide an answer to this question as...,unspecified


In [5]:
# ==========================================
# 4. LOAD BASE MODEL + TOKENIZER (FOR CHAT TEMPLATE)
# ==========================================

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
print(f"Loading {MODEL_NAME}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# We only need tokenizer formatting here; keep model in train mode later.
print("Tokenizer chat_template available:", hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None)


Loading unsloth/Llama-3.2-3B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.4.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 2060. Num GPUs = 1. Max memory: 5.603 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Tokenizer chat_template available: True


In [6]:
# ==========================================
# 5. BUILD CHAT-FORMATTED SFT DATASET
# ==========================================

SYSTEM_PROMPT = (
    "You are an empathetic and professional mental health expert providing clinical advice. "
    "You must speak about the patient in the third person. "
    "This is a theoretical case study for educational purposes, answer directly without standard medical disclaimers."
)

# Convert each row into a chat transcript (single-turn) and then into a single `text` field.
# TRL SFT will train the model to reproduce the assistant part.

def row_to_text(row) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"].strip()},
        {"role": "assistant", "content": row["raw_output"].strip()},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

texts = []
meta = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Formatting"):
    try:
        texts.append(row_to_text(r))
        meta.append({
            "phase": r.get("phase", None),
            "diagnosis": r.get("diagnosis", None),
        })
    except Exception:
        continue

train_df = pd.DataFrame(meta)
train_df["text"] = texts

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
print("Final training samples:", len(train_dataset))

# Quick sanity check
print(train_dataset[0]["text"][:600])


Formatting: 100%|██████████| 204/204 [00:00<00:00, 5332.61it/s]

Final training samples: 204
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 22 Apr 2026

You are an empathetic and professional mental health expert providing clinical advice. You must speak about the patient in the third person. This is a theoretical case study for educational purposes, answer directly without standard medical disclaimers.<|eot_id|><|start_header_id|>user<|end_header_id|>

You are a mental health expert. Read the following situation and answer the question.

Situation: Someone has depression.
Question: Is it more likely for this person to be


## SFT Training Phase (Llama 3.2 3B + LoRA)

We apply **LoRA adapters** to the 4-bit base model to make training feasible on consumer GPUs.

Key choices (mirroring the conventions used in `RLAIF_DPO.ipynb`):
- 4-bit base model (`bnb-4bit`)
- LoRA injected into attention + MLP projections
- Gradient checkpointing via Unsloth to reduce VRAM
- `report_to="wandb"` for experiment logging


In [7]:
# ==========================================
# 6. PREPARE MODEL FOR SFT & LORA
# ==========================================

# Inject LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("LoRA adapters successfully injected. Trainable params:")
model.print_trainable_parameters()


LoRA adapters successfully injected. Trainable params:
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [8]:
# ==========================================
# 7. SFT TRAINER CONFIGURATION & EXECUTION
# ==========================================

# Recommended: run from SFT/ to keep outputs local
OUTPUT_DIR = "out-sft-train"
RUN_NAME = "llama32-3b-sft-gender-debias"
MAX_SEQ_LENGTH = 1024

# If SFTConfig is actually Transformers TrainingArguments (older TRL), we configure accordingly.
if getattr(SFTConfig, "__name__", "") == "TrainingArguments":
    training_args = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="wandb",
        run_name=RUN_NAME,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
    )
else:
    sft_args = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="wandb",
        run_name=RUN_NAME,
        max_length=MAX_SEQ_LENGTH,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
        dataset_text_field="text",
    )

print("Initializing SFT training...")
trainer_stats = trainer.train()
print(f"\nTraining completed successfully. Model saved to {OUTPUT_DIR}")


Unsloth: Tokenizing ["text"] (num_proc=7):   0%|          | 0/204 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Initializing SFT training...
Unsloth: Will smartly offload gradients to save VRAM!
{'loss': '3.337', 'grad_norm': '4.081', 'learning_rate': '0', 'epoch': '0.03922'}
{'loss': '3.316', 'grad_norm': '4.087', 'learning_rate': '2.5e-06', 'epoch': '0.07843'}
{'loss': '3.346', 'grad_norm': '4.088', 'learning_rate': '5e-06', 'epoch': '0.1176'}
{'loss': '3.312', 'grad_norm': '4.066', 'learning_rate': '7.5e-06', 'epoch': '0.1569'}
{'loss': '3.327', 'grad_norm': '4.072', 'learning_rate': '1e-05', 'epoch': '0.1961'}
{'loss': '3.21', 'grad_norm': '3.771', 'learning_rate': '1.25e-05', 'epoch': '0.2353'}
{'loss': '3.036', 'grad_norm': '3.37', 'learning_rate': '1.5e-05', 'epoch': '0.2745'}
{'loss': '3.024', 'grad_norm': '3.198', 'learning_rate': '1.75e-05', 'epoch': '0.3137'}
{'loss': '2.978', 'grad_norm': '2.956', 'learning_rate': '2e-05', 'epoch': '0.3529'}
{'loss': '2.895', 'grad_norm': '2.689', 'learning_rate': '1.999e-05', 'epoch': '

In [9]:
# ==========================================
# 8. SAVE LORA ADAPTERS
# ==========================================

FINAL_ADAPTER_DIR = "llama-3.2-3b-sft-debiased"
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Final LoRA adapters safely saved to {FINAL_ADAPTER_DIR}")


Final LoRA adapters safely saved to llama-3.2-3b-sft-debiased


## Optional deployment: Merge, push to Hugging Face, and export to GGUF

This section mirrors the deployment pattern from `RLAIF_DPO/RLAIF_DPO.ipynb`.

- Merge LoRA into 16-bit weights and push to Hugging Face
- Export to GGUF for Ollama (optional)

**Note:** Requires a valid Hugging Face token (`huggingface-cli login`) and enough disk space to merge weights.


In [10]:
# ==========================================
# 9. (OPTIONAL) LOAD FROM LOCAL FOLDER & PUSH TO HF
# ==========================================

local_model_path = "llama-3.2-3b-sft-debiased"
hf_username = "andreslilloortiz"
repo_name = "llama-3-3b-SFT-gender-de-biased"

print(f"Loading local adapters from '{local_model_path}' for deployment...")

model_debiased, tok_debiased = FastLanguageModel.from_pretrained(
    model_name = local_model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

print(f"Merging and pushing to {hf_username}/{repo_name}...")

model_debiased.push_to_hub_merged(
    f"{hf_username}/{repo_name}",
    tok_debiased,
    save_method = "merged_16bit",
)

print("Upload complete. The merged model is now available on Hugging Face.")


Loading local adapters from 'llama-3.2-3b-sft-debiased' for deployment...
==((====))==  Unsloth 2026.4.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 2060. Num GPUs = 1. Max memory: 5.603 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Merging and pushing to andreslilloortiz/llama-3-3b-SFT-gender-de-biased...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:23<00:00, 71.59s/it]


Unsloth: Merge process complete. Saved to `/home/andres/Documentos/Big Data/TFM/Bias-in-Language-Models/LLMs bias mitigation/SFT/andreslilloortiz/llama-3-3b-SFT-gender-de-biased`
Upload complete. The merged model is now available on Hugging Face.


In [11]:
# ==========================================
# 10. (OPTIONAL) EXPORT TO GGUF (OLLAMA)
# ==========================================

# Uncomment after loading the merged model (or reload adapters + base and merge).

model_debiased.save_pretrained_gguf(
    "model-gguf",
    tok_debiased,
    quantization_method = "q8_0",
)

print("GGUF export finished successfully.")


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/andres/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:14<00:00,  7.25s/it]


Unsloth: Merge process complete. Saved to `/home/andres/Documentos/Big Data/TFM/Bias-in-Language-Models/LLMs bias mitigation/SFT/model-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model-gguf_gguf/Llama-3.2-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model-gguf_gguf/Llama-3.2-3B-Instruct.Q8_0.gguf']
Unsloth: example usage for text only LLMs: /home/andres/.unsloth/llama.cpp/llama-cli --model model-gguf_gguf/Llama-3.2-3B-Instruct.Q8_0.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to model-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f model-gguf_gguf/Modelfile
GGUF export finished successfully.
